<a href="https://colab.research.google.com/github/BIDBhargavipappuri/Python/blob/PAN_NO_Validation/PAN_Validation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

` When I tried to read the CSV file on my desktop in a folder colab was not able to read the file so followed below 2 steps and proceeded further.`

In [9]:
import os


print(os.path.exists(r"C:\Users\bharg\Desktop\Python\PAN_NO\PAN_Data.xlsx"))

False


In [13]:
from google.colab import files
uploaded = files.upload()

Saving PAN_Data.xlsx to PAN_Data (1).xlsx


In [25]:
import pandas as pd  #To Handle data cleansing
import numpy as np   #To detecting sequences
import re            #for validation of PAN number
import string        #filtering non‑alphanumeric character
import os            #reading files from folders
import logging       #tracking invalid PANs


df = pd.read_excel("PAN_Data.xlsx")
print("Data loaded successfully!")
df.head(5)         #Display top 5 columns from CSV file

PAN_COL = "Pan_Numbers"

# 1️⃣ Identify missing PAN values

missing_count = df[PAN_COL].isna().sum()

print("Total rows:", len(df))
print("Missing PAN values:", missing_count)


# 2️⃣ Impute missing PAN values

# We replace missing PANs with a placeholder
df[PAN_COL] = df[PAN_COL].fillna("MISSING_PAN")

# Verify imputation
print("\nMissing PAN values after imputation:",
      df[PAN_COL].isna().sum())

print("\nPreview after imputation:")
print(df.head())

# STEP 2: Detect and Remove Duplicate PAN Numbers

PAN_COL = "Pan_Numbers"

# 1️⃣ Count duplicates BEFORE removal
duplicate_count = df.duplicated(subset=[PAN_COL]).sum()
print("Duplicate PAN count:", duplicate_count)

# Optional: view duplicate rows
if duplicate_count > 0:
    print("\nSample duplicate rows:")
    print(df[df.duplicated(subset=[PAN_COL], keep=False)].head())

# 2️⃣ Remove duplicates (keep the first occurrence)
df = df.drop_duplicates(subset=[PAN_COL], keep="first")

# 3️⃣ Confirm removal
print("\nShape after removing duplicates:", df.shape)

# STEP 3: Remove leading and trailing spaces from PAN numbers

PAN_COL = "Pan_Numbers"

# Convert to string (important if column contains numbers or NaN)
df[PAN_COL] = df[PAN_COL].astype(str)

# Strip spaces before and after the PAN
df[PAN_COL] = df[PAN_COL].str.strip()

print("Spaces removed successfully!")
print(df.head())

# STEP 4: Convert PAN numbers to uppercase

PAN_COL = "Pan_Numbers"

df[PAN_COL] = df[PAN_COL].astype(str).str.upper()

print("PAN numbers converted to uppercase!")
print(df.head())

#step 5 Full PAN Format Validation

PAN_COL = "Pan_Numbers"

# Helper: Check if a string is sequential (ABCDE, BCDEF, 1234, 2345)
def is_sequential(s):
    return all(ord(s[i+1]) - ord(s[i]) == 1 for i in range(len(s)-1))

# Helper: Check if adjacent characters are identical
def has_adjacent_duplicates(s):
    return any(s[i] == s[i+1] for i in range(len(s)-1))

# Main PAN validation function
def validate_pan(pan):
    pan = str(pan).strip().upper()

    # Rule 1: Length must be exactly 10
    if len(pan) != 10:
        return False, "Length not equal to 10"

    first5 = pan[:5]
    next4 = pan[5:9]
    last1 = pan[9]

    # Rule 2: First 5 must be alphabets
    if not first5.isalpha():
        return False, "First 5 chars must be alphabets"

    # Rule 3: No adjacent alphabet duplicates
    if has_adjacent_duplicates(first5):
        return False, "Adjacent alphabet characters identical"

    # Rule 4: First 5 cannot be sequential
    if is_sequential(first5):
        return False, "First 5 alphabets form a sequence"

    # Rule 5: Next 4 must be digits
    if not next4.isdigit():
        return False, "Middle 4 chars must be digits"

    # Rule 6: No adjacent digit duplicates
    if has_adjacent_duplicates(next4):
        return False, "Adjacent digit characters identical"

    # Rule 7: Next 4 digits cannot be sequential
    if is_sequential(next4):
        return False, "Middle 4 digits form a sequence"

    # Rule 8: Last character must be alphabet
    if not last1.isalpha():
        return False, "Last character must be alphabet"

    return True, "Valid PAN"

# APPLY VALIDATION
df["PAN_Valid"], df["Validation_Remark"] = zip(*df[PAN_COL].apply(validate_pan))

print(df.head())

# STEP 6: Separate valid and invalid PANs

valid_pans = df[df["PAN_Valid"] == True].copy()
invalid_pans = df[df["PAN_Valid"] == False].copy()

print("Valid PANs:", len(valid_pans))
print("Invalid PANs:", len(invalid_pans))

print("\nSample valid PANs:")
print(valid_pans.head())

print("\nSample invalid PANs with reasons:")
print(invalid_pans.head())

total = len(df)
valid = df["PAN_Valid"].sum()
invalid = total - valid

print("Total PANs:", total)
print("Valid PANs:", valid)
print("Invalid PANs:", invalid)
print("Validity %:", round(valid/total * 100, 2))
print("Invalid %:", round(invalid/total * 100, 2))

# Count reasons for invalidity
reason_counts = df["Validation_Remark"].value_counts()
print("\nInvalidity Reasons:")
print(reason_counts)



Data loaded successfully!
Total rows: 10000
Missing PAN values: 965

Missing PAN values after imputation: 0

Preview after imputation:
  Pan_Numbers
0  VGLOD3180G
1  PHOXD7232L
2  MGEPH6532A
3  JJCHK4574O
4  XTQIJ2330L
Duplicate PAN count: 972

Sample duplicate rows:
      Pan_Numbers
5022  MISSING_PAN
5027  MISSING_PAN
5033  MISSING_PAN
5043  MISSING_PAN
5057  MISSING_PAN

Shape after removing duplicates: (9028, 1)
Spaces removed successfully!
  Pan_Numbers
0  VGLOD3180G
1  PHOXD7232L
2  MGEPH6532A
3  JJCHK4574O
4  XTQIJ2330L
PAN numbers converted to uppercase!
  Pan_Numbers
0  VGLOD3180G
1  PHOXD7232L
2  MGEPH6532A
3  JJCHK4574O
4  XTQIJ2330L
  Pan_Numbers  PAN_Valid                       Validation_Remark
0  VGLOD3180G       True                               Valid PAN
1  PHOXD7232L       True                               Valid PAN
2  MGEPH6532A       True                               Valid PAN
3  JJCHK4574O      False  Adjacent alphabet characters identical
4  XTQIJ2330L      Fal

In [17]:
df = pd.read_excel("PAN_Data.xlsx")
df["Pan_Numbers"].isna().sum()

np.int64(965)

In [26]:
import pandas as pd
import numpy as np
import re
import string
import os
import logging

df = pd.read_excel("PAN_Data.xlsx")
PAN_COL = "Pan_Numbers"

# ---------------------------
# STEP 1 — Missing Values
# ---------------------------
missing_count = df[PAN_COL].isna().sum()
df[PAN_COL] = df[PAN_COL].fillna("MISSING_PAN")

# ---------------------------
# STEP 2 — Remove Duplicates
# ---------------------------
df = df.drop_duplicates(subset=[PAN_COL], keep="first")

# ---------------------------
# STEP 3 — Strip Spaces
# ---------------------------
df[PAN_COL] = df[PAN_COL].astype(str).str.strip()

# ---------------------------
# STEP 4 — Uppercase
# ---------------------------
df[PAN_COL] = df[PAN_COL].str.upper()

# ---------------------------
# STEP 5 — PAN Validation
# ---------------------------

def is_sequential(s):
    return all(ord(s[i+1]) - ord(s[i]) == 1 for i in range(len(s)-1))

def has_adjacent_duplicates(s):
    return any(s[i] == s[i+1] for i in range(len(s)-1))

def validate_pan(pan):
    pan = str(pan).strip().upper()

    if len(pan) != 10:
        return False, "Length not equal to 10"

    first5 = pan[:5]
    next4 = pan[5:9]
    last1 = pan[9]

    if not first5.isalpha():
        return False, "First 5 chars must be alphabets"

    if has_adjacent_duplicates(first5):
        return False, "Adjacent alphabet characters identical"

    if is_sequential(first5):
        return False, "First 5 alphabets form a sequence"

    if not next4.isdigit():
        return False, "Middle 4 chars must be digits"

    if has_adjacent_duplicates(next4):
        return False, "Adjacent digit characters identical"

    if is_sequential(next4):
        return False, "Middle 4 digits form a sequence"

    if not last1.isalpha():
        return False, "Last character must be alphabet"

    return True, "Valid PAN"

df["PAN_Valid"], df["Validation_Remark"] = zip(*df[PAN_COL].apply(validate_pan))

# ---------------------------
# STEP 6 — Summary Output Only
# ---------------------------

total_records = len(df)
valid_count = df["PAN_Valid"].sum()
invalid_count = total_records - valid_count
missing_or_incomplete = missing_count   # from original dataset

print("Total records processed:", total_records)
print("Total valid PANs:", valid_count)
print("Total invalid PANs:", invalid_count)
print("Total missing or incomplete PANs:", missing_or_incomplete)

Total records processed: 9028
Total valid PANs: 3186
Total invalid PANs: 5842
Total missing or incomplete PANs: 965
